In [1]:
!pip install pandas matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [3]:
base_dir = "."
structured_csv_dir = os.path.join(base_dir, "structured_csv")
os.makedirs(structured_csv_dir, exist_ok=True)

print("structured_csv_dir:", structured_csv_dir)

structured_csv_dir: .\structured_csv


In [4]:
input_file = "../02_cleaning/cleaned_csv/dataset1_cleaned_final.csv"
df = pd.read_csv(input_file)

print("Rows:", len(df))
print(df.columns.tolist())
display(df.head())

Rows: 265
['query', 'video_id', 'title', 'channel', 'duration', 'view_count', 'upload_date', 'url', 'webpage_url', 'query_clean', 'title_clean', 'channel_clean', 'boundary_score', 'sound_score', 'noise_score', 'keep_initial']


,query,video_id,title,channel,duration,view_count,upload_date,url,webpage_url,query_clean,title_clean,channel_clean,boundary_score,sound_score,noise_score,keep_initial
0,bird warning call,dvK-DujvpSY,White bellbird: listen to the world's loudest ...,Guardian News,31.0,9885069.0,NaN,https://www.youtube.com/watch?v=dvK-DujvpSY,NaN,bird warning call,white bellbird listen to the world s loudest b...,guardian news,0,1,0,True
1,bird warning call,cPIIVjkp7k0,Bluebird Alarm Call,Wild Birds Unlimited Macomb,12.0,26065.0,NaN,https://www.youtube.com/watch?v=cPIIVjkp7k0,NaN,bird warning call,bluebird alarm call,wild birds unlimited macomb,1,1,0,True
2,bird warning call,ga_Ybn_K4rc,Chickadee alarm call (16 alarm call at the end!),Bird Feeder Hub,29.0,13511.0,NaN,https://www.youtube.com/watch?v=ga_Ybn_K4rc,NaN,bird warning call,chickadee alarm call 16 alarm call at the end,bird feeder hub,1,1,0,True
3,bird warning call,uuxTZN98rHI,"5 bird alarm calls: blackbird, wren, blackcap,...",Watch the Birdie,96.0,30788.0,NaN,https://www.youtube.com/watch?v=uuxTZN98rHI,NaN,bird warning call,5 bird alarm calls blackbird wren blackcap thr...,watch the birdie,1,2,0,True
4,bird warning call,2aFIujmqJfI,Bird Calls: Is That An Alarm?,NatureMentor,369.0,2894.0,NaN,https://www.youtube.com/watch?v=2aFIujmqJfI,NaN,bird warning call,bird calls is that an alarm,naturementor,1,2,0,True


In [5]:
def normalize_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"[\n\r\t]+", " ", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

df["query_clean"] = df["query"].fillna("").astype(str).apply(normalize_text)
df["title_clean"] = df["title"].fillna("").astype(str).apply(normalize_text)
df["channel_clean"] = df["channel"].fillna("").astype(str).apply(normalize_text)

display(df[["query", "title", "title_clean"]].head())

,query,title,title_clean
0,bird warning call,White bellbird: listen to the world's loudest ...,white bellbird listen to the world s loudest b...
1,bird warning call,Bluebird Alarm Call,bluebird alarm call
2,bird warning call,Chickadee alarm call (16 alarm call at the end!),chickadee alarm call 16 alarm call at the end
3,bird warning call,"5 bird alarm calls: blackbird, wren, blackcap,...",5 bird alarm calls blackbird wren blackcap thr...
4,bird warning call,Bird Calls: Is That An Alarm?,bird calls is that an alarm


In [6]:
warning_terms = ["warning", "warn", "alarm", "alert"]
territorial_terms = ["territorial", "territory", "defend", "defensive"]
distress_terms = ["distress", "panic", "scare", "fear", "danger"]
predator_terms = ["predator", "predators", "attack", "threat", "aggressive", "hawk", "owl", "eagle"]
urban_terms = ["urban", "sub urban", "suburban", "city", "street", "town"]
sound_terms = ["call", "calls", "sound", "sounds", "cry", "cries", "scream", "screams", "vocal", "vocalization", "noise", "noises"]

bird_terms = [
    "bird", "birds", "owl", "owls", "eagle", "eagles", "hawk", "hawks",
    "crow", "crows", "pigeon", "pigeons", "sparrow", "sparrows",
    "parrot", "parrots", "seagull", "gull", "gulls", "starling", "starlings"
]

mammal_terms = [
    "fox", "foxes", "wolf", "wolves", "coyote", "coyotes", "dog", "dogs",
    "cat", "cats", "squirrel", "squirrels", "rat", "rats", "mouse", "mice",
    "deer", "monkey", "monkeys", "lion", "leopard", "tiger", "bear", "bears"
]

In [7]:
def has_any_term(text, terms):
    text = str(text)
    return any(term in text for term in terms)

def matched_terms(text, terms):
    text = str(text)
    return [term for term in terms if term in text]

full_text = df["query_clean"] + " " + df["title_clean"]

df["has_warning_term"] = full_text.apply(lambda x: has_any_term(x, warning_terms))
df["has_territorial_term"] = full_text.apply(lambda x: has_any_term(x, territorial_terms))
df["has_distress_term"] = full_text.apply(lambda x: has_any_term(x, distress_terms))
df["has_predator_term"] = full_text.apply(lambda x: has_any_term(x, predator_terms))
df["has_urban_term"] = full_text.apply(lambda x: has_any_term(x, urban_terms))
df["has_sound_term"] = full_text.apply(lambda x: has_any_term(x, sound_terms))

display(df[[
    "title",
    "has_warning_term",
    "has_territorial_term",
    "has_distress_term",
    "has_predator_term",
    "has_urban_term",
    "has_sound_term"
]].head(15))

,title,has_warning_term,has_territorial_term,has_distress_term,has_predator_term,has_urban_term,has_sound_term
0,White bellbird: listen to the world's loudest ...,True,False,False,False,False,True
1,Bluebird Alarm Call,True,False,False,False,False,True
2,Chickadee alarm call (16 alarm call at the end!),True,False,False,False,False,True
3,"5 bird alarm calls: blackbird, wren, blackcap,...",True,False,False,False,False,True
4,Bird Calls: Is That An Alarm?,True,False,False,False,False,True
5,Blackbird alarm call,True,False,False,False,False,True
6,Decoding bird calls: What they are REALLY Saying?,True,False,False,False,False,True
7,Loud American Robin Alarm Warning Call - Robin...,True,False,False,False,False,True
8,Anti Birds Repellent Sound - calls of birds of...,True,False,True,False,False,True
9,Cardinal Bird-Calling Sounds promo,True,False,False,False,False,True


In [8]:
def assign_sound_logic_group(row):
    if row["has_warning_term"] and row["has_territorial_term"]:
        return "warning_and_territorial"
    elif row["has_warning_term"]:
        return "warning_alarm"
    elif row["has_territorial_term"]:
        return "territorial"
    elif row["has_distress_term"]:
        return "distress_defensive"
    elif row["has_predator_term"]:
        return "predator_threat"
    else:
        return "general_animal_sound"

def assign_boundary_logic_group(row):
    if row["has_territorial_term"]:
        return "space_claiming"
    elif row["has_warning_term"] or row["has_distress_term"]:
        return "alerting_or_repelling"
    elif row["has_predator_term"]:
        return "threat_response"
    elif row["has_urban_term"]:
        return "urban_animal_presence"
    else:
        return "weak_boundary_signal"

df["sound_logic_group"] = df.apply(assign_sound_logic_group, axis=1)
df["boundary_logic_group"] = df.apply(assign_boundary_logic_group, axis=1)

display(df[["title", "sound_logic_group", "boundary_logic_group"]].head(20))

,title,sound_logic_group,boundary_logic_group
0,White bellbird: listen to the world's loudest ...,warning_alarm,alerting_or_repelling
1,Bluebird Alarm Call,warning_alarm,alerting_or_repelling
2,Chickadee alarm call (16 alarm call at the end!),warning_alarm,alerting_or_repelling
3,"5 bird alarm calls: blackbird, wren, blackcap,...",warning_alarm,alerting_or_repelling
4,Bird Calls: Is That An Alarm?,warning_alarm,alerting_or_repelling
5,Blackbird alarm call,warning_alarm,alerting_or_repelling
6,Decoding bird calls: What they are REALLY Saying?,warning_alarm,alerting_or_repelling
7,Loud American Robin Alarm Warning Call - Robin...,warning_alarm,alerting_or_repelling
8,Anti Birds Repellent Sound - calls of birds of...,warning_alarm,alerting_or_repelling
9,Cardinal Bird-Calling Sounds promo,warning_alarm,alerting_or_repelling


In [9]:
def assign_animal_type_hint(text):
    text = str(text)

    bird_hit = has_any_term(text, bird_terms)
    mammal_hit = has_any_term(text, mammal_terms)

    if bird_hit and mammal_hit:
        return "mixed"
    elif bird_hit:
        return "bird"
    elif mammal_hit:
        return "mammal"
    elif "animal" in text or "wildlife" in text:
        return "general_animal"
    else:
        return "unknown"

def assign_urban_reference(text):
    return "urban_explicit" if has_any_term(text, urban_terms) else "urban_unspecified"

df["animal_type_hint"] = full_text.apply(assign_animal_type_hint)
df["urban_reference"] = full_text.apply(assign_urban_reference)

display(df[["title", "animal_type_hint", "urban_reference"]].head(20))

,title,animal_type_hint,urban_reference
0,White bellbird: listen to the world's loudest ...,bird,urban_unspecified
1,Bluebird Alarm Call,bird,urban_unspecified
2,Chickadee alarm call (16 alarm call at the end!),bird,urban_unspecified
3,"5 bird alarm calls: blackbird, wren, blackcap,...",bird,urban_unspecified
4,Bird Calls: Is That An Alarm?,bird,urban_unspecified
5,Blackbird alarm call,bird,urban_unspecified
6,Decoding bird calls: What they are REALLY Saying?,bird,urban_unspecified
7,Loud American Robin Alarm Warning Call - Robin...,bird,urban_unspecified
8,Anti Birds Repellent Sound - calls of birds of...,bird,urban_unspecified
9,Cardinal Bird-Calling Sounds promo,bird,urban_unspecified


In [10]:
def assign_boundary_strength(row):
    score = 0

    if row["has_warning_term"]:
        score += 2
    if row["has_territorial_term"]:
        score += 2
    if row["has_distress_term"]:
        score += 1
    if row["has_predator_term"]:
        score += 1
    if row["has_urban_term"]:
        score += 1

    if score >= 4:
        return "high"
    elif score >= 2:
        return "medium"
    else:
        return "low"

df["boundary_strength"] = df.apply(assign_boundary_strength, axis=1)

display(df[["title", "boundary_strength"]].head(20))

,title,boundary_strength
0,White bellbird: listen to the world's loudest ...,medium
1,Bluebird Alarm Call,medium
2,Chickadee alarm call (16 alarm call at the end!),medium
3,"5 bird alarm calls: blackbird, wren, blackcap,...",medium
4,Bird Calls: Is That An Alarm?,medium
5,Blackbird alarm call,medium
6,Decoding bird calls: What they are REALLY Saying?,medium
7,Loud American Robin Alarm Warning Call - Robin...,medium
8,Anti Birds Repellent Sound - calls of birds of...,medium
9,Cardinal Bird-Calling Sounds promo,medium


In [11]:
all_logic_terms = warning_terms + territorial_terms + distress_terms + predator_terms + urban_terms + sound_terms

df["matched_logic_terms"] = full_text.apply(lambda x: ", ".join(sorted(set(matched_terms(x, all_logic_terms)))))

def build_combined_text(row):
    parts = [
        row["query_clean"],
        row["title_clean"],
        row["sound_logic_group"],
        row["boundary_logic_group"],
        row["animal_type_hint"],
        row["urban_reference"],
        row["boundary_strength"],
        row["matched_logic_terms"]
    ]
    return " | ".join([str(p).strip() for p in parts if str(p).strip() != ""])

df["combined_text_for_vectorization"] = df.apply(build_combined_text, axis=1)

display(df[[
    "title",
    "matched_logic_terms",
    "combined_text_for_vectorization"
]].head(10))

,title,matched_logic_terms,combined_text_for_vectorization
0,White bellbird: listen to the world's loudest ...,"call, warn, warning",bird warning call | white bellbird listen to t...
1,Bluebird Alarm Call,"alarm, call, warn, warning",bird warning call | bluebird alarm call | warn...
2,Chickadee alarm call (16 alarm call at the end!),"alarm, call, warn, warning",bird warning call | chickadee alarm call 16 al...
3,"5 bird alarm calls: blackbird, wren, blackcap,...","alarm, call, calls, warn, warning",bird warning call | 5 bird alarm calls blackbi...
4,Bird Calls: Is That An Alarm?,"alarm, call, calls, warn, warning",bird warning call | bird calls is that an alar...
5,Blackbird alarm call,"alarm, call, warn, warning",bird warning call | blackbird alarm call | war...
6,Decoding bird calls: What they are REALLY Saying?,"call, calls, warn, warning",bird warning call | decoding bird calls what t...
7,Loud American Robin Alarm Warning Call - Robin...,"alarm, call, sound, sounds, warn, warning",bird warning call | loud american robin alarm ...
8,Anti Birds Repellent Sound - calls of birds of...,"call, calls, scare, sound, warn, warning",bird warning call | anti birds repellent sound...
9,Cardinal Bird-Calling Sounds promo,"call, sound, sounds, warn, warning",bird warning call | cardinal bird calling soun...


In [12]:
structured_file = os.path.join(structured_csv_dir, "dataset1_structured.csv")
df.to_csv(structured_file, index=False, encoding="utf-8-sig")

print("Saved file:", structured_file)

Saved file: .\structured_csv\dataset1_structured.csv


In [13]:
report_table = df[[
    "query",
    "title",
    "channel",
    "sound_logic_group",
    "boundary_logic_group",
    "animal_type_hint",
    "urban_reference",
    "boundary_strength",
    "matched_logic_terms",
    "combined_text_for_vectorization"
]].copy()

report_file = os.path.join(structured_csv_dir, "dataset1_report_table.csv")
report_table.to_csv(report_file, index=False, encoding="utf-8-sig")

print("Saved file:", report_file)
display(report_table.head())

Saved file: .\structured_csv\dataset1_report_table.csv


,query,title,channel,sound_logic_group,boundary_logic_group,animal_type_hint,urban_reference,boundary_strength,matched_logic_terms,combined_text_for_vectorization
0,bird warning call,White bellbird: listen to the world's loudest ...,Guardian News,warning_alarm,alerting_or_repelling,bird,urban_unspecified,medium,"call, warn, warning",bird warning call | white bellbird listen to t...
1,bird warning call,Bluebird Alarm Call,Wild Birds Unlimited Macomb,warning_alarm,alerting_or_repelling,bird,urban_unspecified,medium,"alarm, call, warn, warning",bird warning call | bluebird alarm call | warn...
2,bird warning call,Chickadee alarm call (16 alarm call at the end!),Bird Feeder Hub,warning_alarm,alerting_or_repelling,bird,urban_unspecified,medium,"alarm, call, warn, warning",bird warning call | chickadee alarm call 16 al...
3,bird warning call,"5 bird alarm calls: blackbird, wren, blackcap,...",Watch the Birdie,warning_alarm,alerting_or_repelling,bird,urban_unspecified,medium,"alarm, call, calls, warn, warning",bird warning call | 5 bird alarm calls blackbi...
4,bird warning call,Bird Calls: Is That An Alarm?,NatureMentor,warning_alarm,alerting_or_repelling,bird,urban_unspecified,medium,"alarm, call, calls, warn, warning",bird warning call | bird calls is that an alar...


In [14]:
print("Rows:", len(df))

print("\nSound logic group:")
print(df["sound_logic_group"].value_counts())

print("\nBoundary logic group:")
print(df["boundary_logic_group"].value_counts())

print("\nAnimal type hint:")
print(df["animal_type_hint"].value_counts())

print("\nBoundary strength:")
print(df["boundary_strength"].value_counts())

Rows: 265

Sound logic group:
sound_logic_group
warning_alarm              176
territorial                 87
warning_and_territorial      2
Name: count, dtype: int64

Boundary logic group:
boundary_logic_group
alerting_or_repelling    176
space_claiming            89
Name: count, dtype: int64

Animal type hint:
animal_type_hint
bird              124
general_animal     72
mammal             65
mixed               4
Name: count, dtype: int64

Boundary strength:
boundary_strength
medium    252
high       13
Name: count, dtype: int64


In [15]:
display(df[[
    "query",
    "title",
    "sound_logic_group",
    "boundary_logic_group",
    "animal_type_hint",
    "urban_reference",
    "boundary_strength"
]].sample(20, random_state=42))

,query,title,sound_logic_group,boundary_logic_group,animal_type_hint,urban_reference,boundary_strength
179,animal warning sound urban,Barking dogs and urban traffic sound effect,warning_alarm,alerting_or_repelling,mammal,urban_explicit,medium
115,alarm call animal,"Alarm Calls, Crying Wolf, Referential & Affere...",warning_alarm,alerting_or_repelling,mammal,urban_unspecified,medium
96,alarm call animal,Monkeys Sound Alarm To Save Deer From A Tiger ...,warning_alarm,alerting_or_repelling,mammal,urban_unspecified,medium
24,bird warning call,Bird making Samsung notification sounds,warning_alarm,alerting_or_repelling,mixed,urban_unspecified,medium
9,bird warning call,Cardinal Bird-Calling Sounds promo,warning_alarm,alerting_or_repelling,bird,urban_unspecified,medium
139,alarm call animal,The Inconvenient Alarm Call,warning_alarm,alerting_or_repelling,general_animal,urban_unspecified,medium
255,territorial animal sound,A minor territorial dispute breaks out between...,territorial,space_claiming,mammal,urban_unspecified,medium
45,bird warning call,Predator Hunting Call - Distressed Woodpecker ...,warning_alarm,alerting_or_repelling,bird,urban_unspecified,high
185,animal warning sound urban,African Goose Honking - Is this how dinos soun...,warning_alarm,alerting_or_repelling,general_animal,urban_explicit,medium
125,alarm call animal,Animal alarm calls of the Sri Lankan dry zone ...,warning_alarm,alerting_or_repelling,general_animal,urban_unspecified,medium
